# Knowledge base (ChromaDB)

In [164]:
from transformers import CLIPModel, CLIPProcessor
from PIL import Image
import torch
import pandas as pd
import chromadb
from tqdm.notebook import tqdm

CSV_PATH = "./knowledge/clean_youtube_watch_log.csv"
THUMBNAIL_PATH = "./knowledge/thumbnails"

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_or_create_collection(name="youtube_videos")


def embed_text(text: str):
    """Embed a text using CLIP"""
    inputs = proc(
        text=[text], return_tensors="pt", truncation=True, padding=True
    )  # WARNING! Truncation may lead to loss of information for long texts
    with torch.no_grad():
        emb = model.get_text_features(**inputs)
    return emb / emb.norm(dim=-1, keepdim=True)  # Normalize

def embed_image(image: Image.Image):
    """Embed an image using CLIP"""
    inputs = proc(images=image, return_tensors="pt")
    with torch.no_grad():
        emb = model.get_image_features(**inputs)
    return emb / emb.norm(dim=-1, keepdim=True)  # Normalize


# Find new videos to eventually add to the collection
df = pd.read_csv(CSV_PATH)
ids_to_add = set(df["video_id"]) - set(collection.get(ids=None)["ids"])
print(f"Found {len(ids_to_add)} new videos to add to the collection.")

for video_id in tqdm(ids_to_add):
    video_data = df[df["video_id"] == video_id].iloc[0]
    title = video_data["video_title"]
    thumbnail = Image.open(f"{THUMBNAIL_PATH}/{video_id}.jpg")

    # Embed text and image, then fuse them (weighted average)
    text_emb = embed_text(title)
    img_emb = embed_image(thumbnail)
    fused_emb = 0.7 * text_emb + 0.3 * img_emb

    # Add embeddings to the collection
    collection.add(
        ids=[video_id],  # identifier for each entry (e.g., video ID)
        embeddings=[fused_emb],  # list of embeddings (fused text + image)
        documents=[title],  # optional: store the original titles as metadata
    )

Found 0 new videos to add to the collection.


0it [00:00, ?it/s]

# Building the Agent (LangChain)

In [165]:
from dataclasses import dataclass
from typing import Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_ollama import ChatOllama
from langgraph.graph.message import add_messages
from PIL import Image
from typing import Literal
from typing import Optional
from langgraph.types import Command
from langchain.agents import AgentState
from langchain.agents import create_agent
from langchain.tools import ToolRuntime, tool
from langchain.messages import ToolMessage
from dataclasses import asdict
from langchain_groq import ChatGroq
from dataclasses import field
from langchain_core.tools import tool, InjectedToolCallId


with open('llm-api-key.txt') as f:
    api_key = f.readline().strip()

# llm = ChatOllama(model="qwen3:0.6b-q4_K_M", temperature=0)
# llm = ChatOllama(model="qwen3.5:2B", temperature=0)
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=api_key,
)
# llm = ChatGoogleGenerativeAI(
#     model="gemini-2.5-flash-lite",
#     temperature=0,
#     api_key=api_key,
#     convert_system_message_to_human=True,
# )

## 1. Data types

In [166]:


@dataclass
class WatchItem:
    user_id: int
    video_id: str
    video_title: str
    timestamp: str      # ISO format

    @property
    def thumbnail(self) -> Image.Image:
        return Image.open(f'./knowledge/thumbnails/{self.video_id}.jpg')

@dataclass
class BiasProfile:
    emotional_tone: Literal["low", "medium", "high"]
    sensationalism: Literal["low", "medium", "high"]
    topical_narrowing: Literal["low", "medium", "high"]
    echo_chamber_effect: Literal["low", "medium", "high"]
    dominant_topics: list[str]
    evidence_titles: list[str]
    overall_profile: str = field(
        metadata={"description": "Some comments from the model explaining why the scores were assigned."}
    )
    confidence: float = field(
        default=0.0,
        metadata={"description": "A confidence score (0-1) indicating how confident the model is in its bias assessment."}
    )

## 2. Tools

In [167]:
@tool
def retrieve_session(user_id: int, sort_asc: bool, limit: int, runtime: ToolRuntime[None, AgentState]) -> Command:
    """
    Loads a user's watch session.

    Args:
        user_id (int): The user ID.
        sort_asc (bool): True if sort from the oldest first, False if sort from the most recent first.
        limit (int): Max number of results.
        runtime (ToolRuntime[None, AgentState]): The tool runtime.

    Returns:
        list[WatchItem]: The user's watch session.
    """
    df = pd.read_csv(CSV_PATH)
    df = df[df['user_id'] == user_id]   # Filter by user ID
    df["watch_date"] = pd.to_datetime(df["watch_date"])
    df = df.sort_values("watch_date", ascending=sort_asc)

    if limit is not None:
        df = df.head(limit)

    result = df.apply(
        lambda row: WatchItem(
            user_id=row['user_id'],
            video_id=row['video_id'],
            video_title=row['video_title'],
            timestamp=row['watch_date']
        ), axis=1).tolist()

    # Update agent state with the retrieved watch history
    return Command(
        update ={
            "watch_history": result,
            "messages": [ToolMessage(
                content="Retrieved watch history: " + str([asdict(r) for r in result]),
                tool_call_id=runtime.tool_call_id
            )],
        },
    )


@tool
def analyze_bias_profile(watch_history: list[WatchItem], runtime: ToolRuntime[None, AgentState]) -> Command:
    """
    Runs a bias analysis on the provided watch history using a sub-agent.

    Args:
        watch_history (list[WatchItem]): The user's watch session.

    Returns:
        Command: Updates the agent state with the bias profile.
    """

    global bias_analyzer_agent
    
    # Invoke sub-agent with the watch history as input
    result = bias_analyzer_agent.invoke(HumanMessage(content="Analyze this watch history for bias: " + str(watch_history)))
    result = result["structured_response"]    # BiasProfile
    
    # Update the state with the bias profile result from the sub-agent
    return Command(
        update = { 
            "bias_profile": result,
            "messages": [ToolMessage(
                content="Analyzed watch history bias: " + str(result),
                tool_call_id=runtime.tool_call_id
            )],
        },
    )

TOOLS = [retrieve_session, analyze_bias_profile]

## 3. Building the graph

In [168]:
# Global state definition for the graph
class AgentState(AgentState):
    user_id: int
    watch_history: list[WatchItem]                          # @dataclass
    bias_profile: Optional[BiasProfile]
    messages: Annotated[list[BaseMessage], add_messages]    # automatically append 


# ---------- Node 1: General-Purpose LLM agent  ----------
SYSTEM_PROMPT = """
You are a YouTube watch-history analysis agent for rabbit hole and bias prevention.

Rules:
- If you need watch-history items, call retrieve_session.
- If the user asks about bias, narrowing, polarization, ideology, sensationalism, clickbait, emotional tone, or rabbit-hole effects, first ensure watch history is available.
- After enough watch history is available for such an analysis, trigger bias analysis by calling analyze_bias_profile.
- Critically evaluate if more information is before responding to the user and invoke tools as needed.
- Always check for state updates from tool calls and use that information in your reasoning.
- For factual questions, answer directly from the retrieved watch history.
- Be cautious and do not overclaim ideology.
"""

agent = create_agent(
    llm,
    tools=TOOLS,
    state_schema=AgentState,
    system_prompt=SYSTEM_PROMPT,
)


# ---------- Node 2: Bias Analysis sub-agent  ----------
BIAS_AGENT_PROMPT = """
Analyze this YouTube watch session for possible rabbit-hole effects.

Return a structured bias profile.

Rules:
- Use only the provided records.
- Be cautious.
- Do not overclaim ideology.
- If evidence is weak, say unclear.
"""

bias_analyzer_agent = create_agent(
    llm,
    state_schema=AgentState,
    system_prompt=BIAS_AGENT_PROMPT,
    response_format=BiasProfile,
)

# 4. Testing the agent
> **⚠️ WARNING: Don't forget to run cell above!**

In [171]:
user_id = "25"
print("Agent 🤖 is ready. Type 'exit' to quit.\n")

while True:
    user = input("\nYou: ").strip()
    if user == '' or user.lower() == "exit":
        break

    prompt = (
        f"The user_id is {user_id}. "
        f"Answer to this question about that user's session: {user}"
    )

    stream = agent.stream(
        {"messages": [HumanMessage(content=prompt)]},
        stream_mode="messages",
        version="v2",
    )

    print("\n\nYou 🙋‍♂️:", user)
    print("\nAgent 🪨: 🤔 ... ")

    for part in stream:
        if part["data"][0].type != "AIMessageChunk" or not part["data"][0].content:
            continue
        print(part["data"][0].content, end="")

print("\n\nAgent 🪨: Okay then you don't like me. Goodbye.")

Agent 🤖 is ready. Type 'exit' to quit.



You 🙋‍♂️: what is the bias analysis of the last 10 videos i have watched? also tell me the titles of those videos

Agent 🪨: 🤔 ... 
The titles of the last 10 videos you have watched are:

1. SkiVel
2. Lords mobile! What are familiars, And how do you use them
3. DILBAR Full Song | Satyameva Jayate | John Abraham Nora Fatehi | Tanishk B Neha Kakkar Ikka Dhvani
4. Peekaboo Playground | Kids Songs | Super Simple Songs
5. Goal # 5 | All Aboard For Global Goals! | Thomas & Friends
6. Party Street | Hi-5 Season 17 Songs of the Week and more Kids Songs
7. The Evil Jungle | Ben 10 | Cartoon Network
8. Powerpuff Girls Was About a Wrestler?! | WHAT THEY GOT RIGHT
9. StoryBots | Learn The Planets In The Solar System | Outer Space Songs For Kids | Netflix Jr
10. 🔴 POCOYO in ENGLISH - Having a Ball 🔴 | Full Episodes | VIDEOS and CARTOONS FOR KIDS

The bias analysis of your watch history suggests a moderate level of emotional tone and topical narrowing, with l